# Space-to-Sea | Maritime Alerts Module
**Hackathon workstream: Vessel Monitoring & Infrastructure Protection**

> ⚠️ ALL VESSEL TRACKS ARE SIMULATED — NOT REAL AIS DATA

Study area: **Gulf of Mannar / Palk Strait, Tamil Nadu, India**

### Kaggle Setup
1. Upload this notebook to [kaggle.com/code](https://kaggle.com/code) → **New Notebook**
2. If you have a custom dataset (real AIS CSV), add it via **Add Data → Your Datasets**
3. Accelerator: for rule-based detection (default) → **None / CPU** is fine
4. If you extend to a sequence model (LSTM anomaly detector) → switch to **GPU T4×2**
5. Run all cells → download `tracks.geojson`, `alerts.json`, `zones.geojson` from the output panel

---

In [ ]:
# ── Install dependencies (Kaggle already has shapely & geopy; kept for safety)
import subprocess, sys
pkgs = ['shapely>=2.0', 'geopy>=2.4', 'pandas>=2.0']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs)
print('Dependencies ready.')

In [ ]:
import json, uuid, math, os
from datetime import datetime, timedelta, timezone
from shapely.geometry import Point, Polygon
from geopy.distance import geodesic
import pandas as pd

# Output dir — Kaggle writes to /kaggle/working/
OUTPUT_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output → {OUTPUT_DIR}')

## Stage 1 — Zone Definitions

In [ ]:
# ── Watch-area polygon (lon, lat GeoJSON convention)
# Rectangular corridor in the Gulf of Mannar, 8.8°N–9.2°N, 78.8°E–79.3°E
# Swap in your actual grid region once Person 1 locks coordinates.
WATCH_AREA = {
    'zone_id': 'ZONE-TN-GULF-01',
    'name': 'Gulf of Mannar Offshore Survey Corridor',
    'coordinates': [
        [78.80, 8.80], [79.30, 8.80], [79.30, 9.20],
        [78.80, 9.20], [78.80, 8.80],  # closed ring
    ],
}

# ── Protected asset: notional offshore substation
ASSET = {
    'asset_id': 'ASSET-TN-SUB-01',
    'name': 'Gulf of Mannar Offshore Substation (SIMULATED)',
    'lat': 9.00, 'lon': 79.05,
    'proximity_radius_km': 2.0,  # ← tune this per asset scale
}

WATCH_POLY  = Polygon([(c[0], c[1]) for c in WATCH_AREA['coordinates']])
ASSET_POINT = (ASSET['lat'], ASSET['lon'])

print('Zone loaded:', WATCH_AREA['name'])
print('Asset:', ASSET['name'])

## Stage 2 — Simulated Vessel Tracks

In [ ]:
def make_track(track_id, waypoints, t0, interval_min=10, label='SIMULATED'):
    rows = []
    for i, (lat, lon) in enumerate(waypoints):
        rows.append({
            'track_id':    track_id,
            'timestamp':   (t0 + timedelta(minutes=i * interval_min)).isoformat(),
            'lat':         lat,
            'lon':         lon,
            'data_source': label,
        })
    return rows

t0 = datetime(2024, 6, 15, 6, 0, 0, tzinfo=timezone.utc)

# TRACK-001: Fishing vessel — enters polygon AND gets close to asset  → both alerts
track_001 = make_track('TRACK-001', [
    (8.70, 78.70),   # outside — SW
    (8.82, 78.85),   # enters polygon
    (8.92, 78.95),   # inside, heading NE
    (8.99, 79.03),   # ~2.5 km from asset
    (9.00, 79.06),   # ~1.1 km from asset  ← ASSET_PROXIMITY_ALERT
    (9.05, 79.15),   # still inside
    (9.18, 79.25),   # near NE exit
    (9.30, 79.40),   # outside — NE
], t0)

# TRACK-002: Cargo vessel — stays OUTSIDE polygon the whole time  → no alerts
track_002 = make_track('TRACK-002', [
    (8.50, 78.50),
    (8.60, 78.70),
    (8.65, 78.90),
    (8.72, 79.10),
    (8.70, 79.40),
    (8.65, 79.60),
], t0, interval_min=15)

# TRACK-003: Survey vessel — enters polygon but stays far from asset  → BOUNDARY only
track_003 = make_track('TRACK-003', [
    (8.78, 79.28),
    (8.85, 79.22),
    (8.95, 79.18),
    (9.10, 79.20),
    (9.18, 79.28),
    (9.25, 79.35),
], t0, interval_min=12)

all_tracks = track_001 + track_002 + track_003
df_tracks  = pd.DataFrame(all_tracks)
print(df_tracks.groupby('track_id').size().rename('points'))
df_tracks.head()

## Stage 3 — Alert Logic (Rule-Based, CPU-Only)

In [ ]:
alerts = []

for _, row in df_tracks.iterrows():
    lat, lon = row['lat'], row['lon']
    pt = Point(lon, lat)   # shapely Point(lon, lat)

    # Rule 1 — Point-in-polygon
    if WATCH_POLY.contains(pt):
        alerts.append({
            'alert_id':              f'ALT-{uuid.uuid4().hex[:8].upper()}',
            'track_id':              row['track_id'],
            'alert_type':            'BOUNDARY_CROSSING_OBSERVED',
            'timestamp':             row['timestamp'],
            'lat':                   lat,
            'lon':                   lon,
            'zone_id':               WATCH_AREA['zone_id'],
            'asset_id':              None,
            'distance_to_asset_km':  round(geodesic((lat, lon), ASSET_POINT).km, 3),
            'data_source':           'SIMULATED',
        })

    # Rule 2 — Asset proximity
    dist = geodesic((lat, lon), ASSET_POINT).km
    if dist <= ASSET['proximity_radius_km']:
        alerts.append({
            'alert_id':             f'ALT-{uuid.uuid4().hex[:8].upper()}',
            'track_id':             row['track_id'],
            'alert_type':           'ASSET_PROXIMITY_ALERT',
            'timestamp':            row['timestamp'],
            'lat':                  lat,
            'lon':                  lon,
            'zone_id':              None,
            'asset_id':             ASSET['asset_id'],
            'distance_to_asset_km': round(dist, 3),
            'proximity_radius_km':  ASSET['proximity_radius_km'],
            'data_source':          'SIMULATED',
        })

df_alerts = pd.DataFrame(alerts)
print(f'Total alerts: {len(alerts)}')
print(df_alerts['alert_type'].value_counts())
df_alerts[['track_id','alert_type','distance_to_asset_km','timestamp']].head(10)

## Stage 4 — Export Output Files

In [ ]:
# ── tracks.geojson ───────────────────────────────────────────────────────────
features = []
for _, r in df_tracks.iterrows():
    features.append({
        'type': 'Feature',
        'geometry': {'type': 'Point', 'coordinates': [r['lon'], r['lat']]},
        'properties': {
            'track_id':    r['track_id'],
            'timestamp':   r['timestamp'],
            'data_source': 'SIMULATED',
        },
    })
tracks_fc = {
    'type': 'FeatureCollection',
    'features': features,
    'metadata': {'data_source': 'SIMULATED', 'generated_by': 'maritime_alerts_kaggle.ipynb'},
}
with open(f'{OUTPUT_DIR}/tracks.geojson', 'w') as f:
    json.dump(tracks_fc, f, indent=2)
print(f'✓ tracks.geojson  ({len(features)} points)')

# ── alerts.json ──────────────────────────────────────────────────────────────
alerts_out = {
    'metadata': {
        'data_source': 'SIMULATED',
        'generated_by': 'maritime_alerts_kaggle.ipynb',
        'total_alerts': len(alerts),
    },
    'alerts': alerts,
}
with open(f'{OUTPUT_DIR}/alerts.json', 'w') as f:
    json.dump(alerts_out, f, indent=2)
print(f'✓ alerts.json  ({len(alerts)} alerts)')

# ── zones.geojson ────────────────────────────────────────────────────────────
zones_fc = {
    'type': 'FeatureCollection',
    'features': [
        {
            'type': 'Feature',
            'geometry': {'type': 'Polygon', 'coordinates': [WATCH_AREA['coordinates']]},
            'properties': {**WATCH_AREA, 'type': 'WATCH_AREA', 'data_source': 'SIMULATED'},
        },
        {
            'type': 'Feature',
            'geometry': {'type': 'Point', 'coordinates': [ASSET['lon'], ASSET['lat']]},
            'properties': {**ASSET, 'type': 'PROTECTED_ASSET', 'data_source': 'SIMULATED'},
        },
    ],
    'metadata': {'data_source': 'SIMULATED'},
}
with open(f'{OUTPUT_DIR}/zones.geojson', 'w') as f:
    json.dump(zones_fc, f, indent=2)
print(f'✓ zones.geojson  (1 polygon + 1 asset point)')

# ── also save CSV for dashboards that prefer flat files
CSV_DIR = OUTPUT_DIR + '/csv'
os.makedirs(CSV_DIR, exist_ok=True)
df_tracks.to_csv(f'{CSV_DIR}/tracks.csv', index=False)
df_alerts.to_csv(f'{CSV_DIR}/alerts.csv', index=False)
print(f'✓ csv/tracks.csv, csv/alerts.csv')
print(f'\nAll files in: {OUTPUT_DIR}')

## Optional Stage 5 — Anomaly Detection with IsolationForest (CPU / Kaggle GPU)

> **Default for this 6-hour demo:** skip this cell — the rule-based stages above are sufficient.
>
> Enable this only if you have enough time budget and want a data-driven anomaly score on top of the rule-based alerts.
>
> **Kaggle GPU note:** IsolationForest itself is CPU-only (scikit-learn). If you replace it with a Transformer / LSTM sequence model, switch the Kaggle accelerator to **GPU T4×2**. The rest of the notebook code stays unchanged — Kaggle mounts data at `/kaggle/input/` and writes output to `/kaggle/working/` automatically.

### How to port to Kaggle GPU (if needed)
1. Upload this notebook: `kaggle.com/code` → **New Notebook** → **File → Import Notebook**
2. Add your dataset: **Add Data → Upload** (the CSV/GeoJSON files or a real AIS CSV)
3. Accelerator: **Settings → Accelerator → GPU T4×2** (free, ~30 GPU-hrs/week)
4. Click **Run All**
5. Download outputs from the **Output** tab

In [ ]:
# ── Uncomment to run IsolationForest anomaly scoring ─────────────────────────
# (CPU-only; no GPU needed for this model)

# from sklearn.ensemble import IsolationForest
# import numpy as np
#
# # Feature engineering: speed proxy + bearing change per track point
# def track_features(df):
#     df = df.copy().sort_values(['track_id', 'timestamp'])
#     df['dlat'] = df.groupby('track_id')['lat'].diff().fillna(0)
#     df['dlon'] = df.groupby('track_id')['lon'].diff().fillna(0)
#     df['step_dist'] = np.sqrt(df['dlat']**2 + df['dlon']**2)
#     df['bearing']   = np.degrees(np.arctan2(df['dlon'], df['dlat'])) % 360
#     return df[['step_dist', 'bearing', 'lat', 'lon']].fillna(0)
#
# X = track_features(df_tracks)
# clf = IsolationForest(n_estimators=100, contamination=0.1, random_state=42)
# df_tracks['anomaly_score'] = clf.fit_predict(X)   # -1 = anomalous
# df_tracks['is_anomaly']    = df_tracks['anomaly_score'] == -1
# print(df_tracks.groupby('track_id')['is_anomaly'].sum())
# df_tracks.to_csv(f'{CSV_DIR}/tracks_with_anomaly.csv', index=False)
# print('✓ tracks_with_anomaly.csv')

---
## Simulated vs. Production

| Component | This demo | Production |
|---|---|---|
| Vessel tracks | Handcrafted waypoints, `SIMULATED` label | Real AIS feed (MarineTraffic / ORBCOMM API) or SAR-derived tracks |
| Watch-area polygon | Static GeoJSON constant | Loaded from PostGIS / fleet-management system |
| Asset locations | Single hardcoded point | Asset registry API (dynamic, with buffer zones) |
| Alert routing | `alerts.json` file | Pub/Sub queue → dashboard websocket |
| Anomaly detection | `IsolationForest` (optional stub) | Trained sequence model (LSTM/Transformer) on historical AIS |
| Kaggle GPU needed? | **No** — rule-based is CPU-only | Only for sequence model training |

> **Explicitly excluded from this module:** AIS-association/matching, nationality inference, real-time satellite tasking, hostile-intent classification.